In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
PROJECT_ROOT = "/content/drive/MyDrive/Semiconductor-AnomalyDetection"
os.chdir(PROJECT_ROOT)

print(f"current path is: {os.getcwd()}")

current path is: /content/drive/MyDrive/Semiconductor-AnomalyDetection


In [1]:
import pandas as pd
import numpy as np
import os

In [6]:
!ls

app   models	 README.md  requirements.txt  src
data  notebooks  REPORT.md  results	      visualization


In [33]:
data_path = "data/raw/secom.data"
label_path = "data/raw/secom_labels.data"

if os.path.exists(data_path) and os.path.exists(label_path):
  # df_data_preview = pd.read_csv(data_path, sep=r'\s+', header=None, nrows=5) #공백기준 쪼개서 일기

  #1. read features (change col name)
  features = pd.read_csv(data_path, sep= r'\s+', header = None)
  features.columns = [f"feature_{i}" for i in range(features.shape[1])]
  print(features.shape)
  #2. read labels
  labels = pd.read_csv(label_path, sep=r'\s+', header=None, names = ['target', 'timestamp'])
  print(labels['timestamp'].dtype) # object type (just text)
  print(labels.shape)
  print(labels.head())
  #3. merge two dataset
  df = pd.concat([features, labels], axis = 1)
  print(df.shape)
  #4.change timestamp data type to datetime to sort
  df['timestamp'] = pd.to_datetime(df['timestamp'], format = '%d/%m/%Y %H:%M:%S')
  df = df.sort_values('timestamp').reset_index(drop = True)
  print(f"initial time: {df['timestamp'].min()}")
  print(f"end time: {df['timestamp'].max()}")
  print(df.head())
  #5.time-based train,test data split to prevent data leakage
  split_idx = int(len(df) * 0.8)
  #deep copy
  train_df = df.iloc[:split_idx].copy()
  test_df = df.iloc[split_idx:].copy()
  print(f"Train set: {train_df['timestamp'].min()} ~ {train_df['timestamp'].max()} ({len(train_df)}data)")
  print(f"Test set: {test_df['timestamp'].min()} ~ {test_df['timestamp'].max()} ({len(test_df)}data)")
else:
  print("can't find path")




(1567, 590)
object
(1567, 2)
   target            timestamp
0      -1  19/07/2008 11:55:00
1      -1  19/07/2008 12:32:00
2       1  19/07/2008 13:17:00
3      -1  19/07/2008 14:43:00
4      -1  19/07/2008 15:22:00
(1567, 592)
initial time: 2008-07-19 11:55:00
end time: 2008-10-17 06:07:00
   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0    3030.93    2564.00  2187.7333  1411.1265     1.3602      100.0   
1    3095.78    2465.14  2230.4222  1463.6606     0.8294      100.0   
2    2932.61    2559.94  2186.4111  1698.0172     1.5102      100.0   
3    2988.72    2479.90  2199.0333   909.7926     1.3204      100.0   
4    3032.24    2502.87  2233.3667  1326.5200     1.5334      100.0   

   feature_6  feature_7  feature_8  feature_9  ...  feature_582  feature_583  \
0    97.6133     0.1242     1.5005     0.0162  ...       0.5005       0.0118   
1   102.3433     0.1247     1.4966    -0.0005  ...       0.5019       0.0223   
2    95.4878     0.1241     1.4436     0.0

In [38]:
#two different imputation experiment

#version1 filling nan with median of each sensors
sensor_cols = [f'feature_{i}' for i in range(590)]
median_of_each_sensors = train_df[sensor_cols].median()
print(median_of_each_sensors)
# 2. fill nan with median of each sensors
x_train1 = train_df[sensor_cols].fillna(median_of_each_sensors)
y_train1 = train_df['target']

# 3. fill nan of test data set with median of train dataset
x_test1 = test_df[sensor_cols].fillna(median_of_each_sensors)
y_test1 = test_df['target']

# print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)


print(f"number of null values in train dataset: {x_train1.isnull().sum().sum()}")
print(f"number of null values in test dataset: {x_test1.isnull().sum().sum()}")

feature_0      3012.0900
feature_1      2498.8400
feature_2      2200.1333
feature_3      1288.0857
feature_4         1.2981
                 ...    
feature_585       2.7280
feature_586       0.0210
feature_587       0.0144
feature_588       0.0045
feature_589      68.7444
Length: 590, dtype: float64
number of null values in train dataset: 0
number of null values in test dataset: 0


In [40]:
#version2 filling nan using knn
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=3)

# 1. only use traindataset using knn
sensor_cols = [f"feature_{i}" for i in range(590)]
imputer.fit(train_df[sensor_cols])

# 3. fill in na with knn using train dataset (transform)
x_train2_raw = imputer.transform(train_df[sensor_cols])
x_train2 = pd.DataFrame(x_train2_raw, columns=sensor_cols)
y_train2 = train_df['target'].reset_index(drop=True)
print(x_train2.head())
x_test2_raw = imputer.transform(test_df[sensor_cols])
x_test2 = pd.DataFrame(x_test2_raw, columns=sensor_cols)
y_test_B = test_df['target'].reset_index(drop=True)

print(f"number of null values in train dataset: {x_train2.isnull().sum().sum()}개")
print(f"number of null values in test dataset: {x_test2.isnull().sum().sum()}개")

   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0    3030.93    2564.00  2187.7333  1411.1265     1.3602      100.0   
1    3095.78    2465.14  2230.4222  1463.6606     0.8294      100.0   
2    2932.61    2559.94  2186.4111  1698.0172     1.5102      100.0   
3    2988.72    2479.90  2199.0333   909.7926     1.3204      100.0   
4    3032.24    2502.87  2233.3667  1326.5200     1.5334      100.0   

   feature_6  feature_7  feature_8  feature_9  ...  feature_580  feature_581  \
0    97.6133     0.1242     1.5005     0.0162  ...       0.0056    80.264067   
1   102.3433     0.1247     1.4966    -0.0005  ...       0.0060   208.204500   
2    95.4878     0.1241     1.4436     0.0041  ...       0.0148    82.860200   
3   104.2367     0.1217     1.4882    -0.0124  ...       0.0044    73.843200   
4   100.3967     0.1235     1.5031    -0.0031  ...       0.0049   156.185000   

   feature_582  feature_583  feature_584  feature_585  feature_586  \
0       0.5005       0

**Code Timeline**

[loaded raw data] -> [sorted based on timestamp] -> [initialized index] -> [time-based split to prevent data leakage] (80% -> past data, 20% -> most recent data)